Training the model

In [1]:
import pandas as pd
from pathlib import Path

aligned_dir = 'Data/processed/aligned'
stock = 'Reliance_aligned.csv'
reliance_df = pd.read_csv(Path(aligned_dir).joinpath(stock))


print(reliance_df.shape)
reliance_df.head()

(733, 8)


,news_id,headline,news_time,event_date,close_T,ret_1d,ret_2d,ret_3d
0,706cb401-2b3f-44f8-ab10-ae4f47c7feae,"Stocks to Watch Today: SBI, Aurobindo Pharma, ...",2026-02-09 02:07:00+05:30,2026-02-09,1461.599976,-0.002121,0.004858,-0.008689
1,42199003-0738-4a7b-9ca0-278b685e220f,Mukesh Ambani credits stable leadership under ...,2026-02-04 18:17:00+05:30,2026-02-05,1443.400024,0.005127,0.012609,0.010461
2,6105b445-a073-4677-8007-4249f99673e4,ONGC in talks with ExxonMobil for joint bids i...,2026-01-28 18:46:00+05:30,2026-01-29,1391.000000,0.003163,-0.000431,0.033142
3,5d1732d2-19ab-49c3-bf29-6cd360821066,"Goldman Sachs buys minor stake in Axis Bank, V...",2026-01-27 21:05:00+05:30,2026-01-28,1396.699951,-0.004081,-0.000931,-0.004511
4,04429331-0de4-4a90-88af-56c80256f1d9,Buy Reliance Industries; target of Rs 1750: Mo...,2026-01-21 12:59:00+05:30,2026-01-21,1404.599976,-0.001495,-0.013171,-0.017158


In [2]:
X = reliance_df["headline"]
y = reliance_df["ret_1d"]

In [3]:
# Splitting - not using sklearn's train_test_split - bcoz of shuffling
split = int(len(X)*0.85)

X_train = X[:split]
X_val = X[split:]

y_train = y[:split]
y_val = y[split:]

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_tokens = tokenizer(
    X_train.tolist(),
    padding = True,
    truncation = True,
    max_length = 128,
    return_tensors = "pt" # Return pytorch tensors
)

val_tokens = tokenizer(
    X_val.tolist(),
    padding = True,
    truncation = True,
    max_length = 128,
    return_tensors = "pt" # Return pytorch tensors
)

In [5]:
# Converting the labels to tensors
import torch

y_train_tensors = torch.tensor(y_train.values, dtype=torch.float32)
y_val_tensors = torch.tensor(y_val.values, dtype=torch.float32)

In [22]:
# Making the class for loading data in batches

class NewsDataLoader(torch.utils.data.DataLoader):
    def __init__(self, encodings, labels): # inputs -> tokens and labels
        self.encodings = encodings
        self.labels = labels
    
    def __get_item__(self, idx):
        item = {k : v[idx] for k, v in self.encodings.items()}
        '''the .items() returns a dictionary with following keys values - 
        input_ids - [[token_ids of 1st article], [...],...] each list is the token ids for each article
        attention_masks - [[attention_mask for 1st article], [...],...] each list is the attention mask meaning which of the token_ids is important for each article 
        '''
        item["label"] = self.labels[idx]
        return item
    
    def __len__(self):
        return len(self.labels)

In [8]:
# Instantiate the dataloader
train_dataset = NewsDataLoader(train_tokens, y_train_tensors)
val_dataset   = NewsDataLoader(val_tokens, y_val_tensors)

In [9]:
# Loading the model
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=1,
    problem_type="regression")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
device = torch.device("cuda")

device

device(type='cuda')

In [11]:
# Setting training arguments
from transformers import TrainingArguments

training_args = TrainingArguments( 
    output_dir="/results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=10, # Number of steps after which the results are logged
    eval_strategy="epoch" # Evaluate after every epoch
)

In [12]:
# Instanstiating a trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [21]:
trainer.train()

TypeError: 'NewsDataLoader' object is not subscriptable